In [ ]:
# Removendo mensagens de erro:
import warnings

warnings.filterwarnings(
    "ignore",
    message=r".*is_datetime64tz_dtype is deprecated.*",
    category=DeprecationWarning,
)
# Limpando outputs!
from IPython.display import clear_output

## 1. Extraindo os Dados:

#### Importando bibliotecas essenciais para a Análise de Dados:

In [ ]:
# Importando bibliotecas:
from __future__ import annotations                          # Adia a avaliação das type hints (permite "forward references", ajuda a evitar imports circulares e melhora performance em alguns casos)...
from datetime import time, timedelta, datetime              # Corrigindo os horários e as datas...
import pyarrow.parquet as pq                                # PyArrow para trabalhar com Parquet (formato colunar compactado muito usado em data analytics)...
from typing import Optional                                 # Tipagem: indica que um valor pode ser do tipo X ou None (ex.: Optional[int])...
from pathlib import Path                                    # Pegando o endereço do arquivo...
from glob import glob                                       # Busca de arquivos por padrões...
import pandas as pd                                         # Análise e manipulação de dados...
import numpy as np                                          # Tratamento de Dados...
import warnings                                             # Controle de avisos...
import sqlite3                                              # Biblioteca para criar a base de dados em Data Warehouse...
import os                                                   # Operações com arquivos e sistema...

pd.set_option('mode.chained_assignment', None)              # Desativa avisos de atribuição encadeada...
warnings.filterwarnings("ignore", category=FutureWarning)   # Oculta avisos de futura depreciação...


#### Consumindo os Dados dos arquivos Parquet:

In [ ]:
# Caminhos das pastas com os arquivos '.parquet':
pasta_ativos = R"..\Base de Dados Original\parquet\Ativos\Ativos.parquet"
pasta_desligados = R"..\Base de Dados Original\parquet\Desligados\Desligados.parquet"
pasta_mensalistas = R"..\Base de Dados Original\parquet\Mensalistas"
pasta_gptw = R"..\Base de Dados Original\parquet\GPTW\GPTW.parquet"
pasta_executivos = R"..\Base de Dados Original\parquet\Executivos"
pasta_horistas = R"..\Base de Dados Original\parquet\Horistas"
pasta_movimentacoes = R"..\Base de Dados Original\parquet\Movimentacoes\Movimentacoes.parquet"

# Lista todos os arquivos '.parquet' em cada pasta:
arquivos_parquet_executivos = glob(os.path.join(pasta_executivos, "*.parquet"))
arquivos_parquet_mensalistas = glob(os.path.join(pasta_mensalistas, "*.parquet"))
arquivos_parquet_horistas = glob(os.path.join(pasta_horistas, "*.parquet"))

# Carrega os arquivos e adiciona a coluna 'Cargo':
tables = [
    pd.read_parquet(caminho).assign(Cargo=cargo)
    for arquivos, cargo in zip([arquivos_parquet_executivos, arquivos_parquet_mensalistas, arquivos_parquet_horistas], ['E', 'M', 'H'])
    for caminho in arquivos
]

# Carrega o arquivo para dentro do Data Frame:
df_desligados = pd.read_parquet(pasta_desligados)
df_ativos = pd.read_parquet(pasta_ativos)
df_gptw = pd.read_parquet(pasta_gptw)
df_movimentacoes = pd.read_parquet(pasta_movimentacoes)

In [ ]:
# === Resumo final ===
total_linhas = sum(len(df) for df in tables)
print("\n📊 RESUMO GERAL")
print(f"Arquivos executivos: {len(arquivos_parquet_executivos)}")
print(f"Arquivos mensalistas: {len(arquivos_parquet_mensalistas)}")
print(f"Arquivos horistas: {len(arquivos_parquet_horistas)}")
print(f"Total de DataFrames lidos: {len(tables)}")
print(f"Total de linhas combinadas: {total_linhas:,}")

## 2. Tratando os Dados:

#### Definição das Funções para Transformação e Tratamento de Dados:
1. **Renomear Colunas:**  
   Função que renomeia as colunas do DataFrame com base em um dicionário de mapeamento, garantindo padronização e clareza nos nomes.
2. **Mover Colunas:**  
   Função que move as colunas do DataFrame com base em um dicionário de mapeamento, garantindo padronização e clareza nos nomes.

In [ ]:
# Função para renomear coluna:
def renomear_coluna(df, col, newname):
    df = df.rename(columns={col: newname})
    return df

# Função para mover coluna
def mover_coluna(df, col, nova_posicao):
    colunas = list(df.columns)
    colunas.insert(nova_posicao, colunas.pop(colunas.index(col)))
    return df[colunas]

#### Aplicando Funções a lista de tabelas:

In [ ]:
# Aplicando a função a todas as colunas de todas as tabelas:
for i in range(len(tables)):
    clear_output(wait=True)
    print(f"Processando tabela {i+1}/{len(tables)}", end="", flush=True)
    tables[i] = renomear_coluna(tables[i], tables[i].columns[0], 'Cadastro')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[1], 'Coletor')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[2], 'Descricao Coletor')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[3], 'Nome')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[4], 'Filial')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[5], 'Data')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[6], 'Hora')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[7], 'Acesso')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[8], 'Dir')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[9], 'Tipo')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[10], 'Uso Acesso')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[11], 'Planta')
#'Cadastro', 'Nome', 'Filial', 'Data', 'Hora', 'Dir', 'Tipo', 'Acesso', 'Coletor', 'Planta', 'Descricao Coletor', 'Uso Acesso', 'Cargo'...

# Aplicando a função de renomear o nome da coluna nas colunas da tabela 'df_ativos':
df_ativos = renomear_coluna(df_ativos, df_ativos.columns[0], 'Cadastro')
df_ativos = renomear_coluna(df_ativos, df_ativos.columns[1], 'Cargo')
df_ativos = renomear_coluna(df_ativos, df_ativos.columns[2], 'Nome completo')
df_ativos = renomear_coluna(df_ativos, df_ativos.columns[3], 'Heranca')
df_ativos = renomear_coluna(df_ativos, df_ativos.columns[4], 'Cargo Especifico')
df_ativos = renomear_coluna(df_ativos, df_ativos.columns[5], 'Expatriado')
df_ativos = renomear_coluna(df_ativos, df_ativos.columns[6], 'Departamento')
df_ativos = renomear_coluna(df_ativos, df_ativos.columns[7], 'Área')
df_ativos = renomear_coluna(df_ativos, df_ativos.columns[8], 'Local')

# Aplicando a função de renomear o nome da coluna 'Departamento' da tabela 'df_desligados':
df_desligados = mover_coluna(df_desligados, 'Diretoria RH', 2)
df_desligados = renomear_coluna(df_desligados, df_desligados.columns[0], 'Cadastro')
df_desligados = renomear_coluna(df_desligados, df_desligados.columns[2], 'Heranca')
df_desligados = renomear_coluna(df_desligados, df_desligados.columns[3], 'Cargo Especifico')
df_desligados = renomear_coluna(df_desligados, df_desligados.columns[4], 'Departamento')
df_desligados = renomear_coluna(df_desligados, df_desligados.columns[5], 'Área')
df_desligados = renomear_coluna(df_desligados, df_desligados.columns[6], 'Local')

# Aplicando a função de renomear o nome da coluna nas colunas da tabela 'df_gptw':
df_gptw = renomear_coluna(df_gptw, df_gptw.columns[0], 'Área')

# Aplicando a função de renomear o nome da coluna nas colunas da tabela 'df_movimentacoes':
df_movimentacoes = renomear_coluna(df_movimentacoes, df_movimentacoes.columns[0], 'Cadastro')

#### Concatenar todas as tabelas da lista em um único Data Frame:

In [ ]:
# Concatenando todas as tabelas em um único DataFrame:
df_ronda = pd.concat(tables, ignore_index=True)

# Ordenando as colunas na ordem desejada:
df_ronda = df_ronda[['Cadastro', 'Nome', 'Filial', 'Data', 'Hora', 'Dir', 'Tipo', 'Acesso', 'Planta', 'Coletor', 'Descricao Coletor', 'Uso Acesso', 'Cargo']]
df_movimentacoes = df_movimentacoes[
    ['Cadastro', 'Abreviação UO antiga', 'Desc. UO antiga', 'Desc. Cargo antigo',
    'Abreviação UO', 'Desc. UO', 'Desc. Cargo', 'Tipo da Movimentação', 'Data de alteração']
]
df_ronda = df_ronda.drop(df_ronda.index[-1]) # Remover última linha de total...

#### Diferenciando as pessoas que foram demitidas ou que não serão analisadas e inserindo todos os df_ativos delas:

In [ ]:
# Criando toda a função para encontrar as pessoas desligadas não registradas nas bases 'df_ativos' e 'df_desligados':
TARGET_COLS = [
    "Cadastro", "Cargo",
    "Nome completo", "Heranca", "Cargo Especifico",
    "Departamento", "Área", "Local", "Data_Adm", "Data_Dem"
]

def _to_dt(s, dayfirst=True):
    return pd.to_datetime(s, errors="coerce", dayfirst=dayfirst)

def _coalesce(a, b):
    return a.where(~a.isna(), b)

def _fix_cols_ativos(df: pd.DataFrame) -> pd.DataFrame:
    """
    Garante que df_ativos tenha:
      - colunas únicas (sem duplicadas)
      - 'Cargo' e 'Cargo Especifico' separados (se vierem duplicados como 'Cargo Especifico')
    Regra aplicada:
      - Se houver 2+ 'Cargo Especifico' e não existir 'Cargo', renomeia a 1ª ocorrência para 'Cargo'
      - Se houver só 1 'Cargo Especifico' e não existir 'Cargo', cria 'Cargo' copiando 'Cargo Especifico'
    """
    df = df.copy()
    df.columns = df.columns.astype(str).str.strip()

    cols = pd.Series(df.columns)
    mask_ce = cols.eq("Cargo Especifico")

    if "Cargo" not in cols.values:
        if mask_ce.sum() >= 2:
            idxs = np.flatnonzero(mask_ce.values)
            cols.iloc[idxs[0]] = "Cargo"  # 1ª ocorrência vira Cargo
        elif mask_ce.sum() == 1:
            # cria Cargo copiando Cargo Especifico (melhor do que NA)
            df["Cargo"] = df["Cargo Especifico"]

    df.columns = cols.tolist()

    # remove duplicadas (se ainda restar alguma)
    df = df.loc[:, ~df.columns.duplicated(keep="last")]
    return df

def _dedup_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = df.columns.astype(str).str.strip()
    return df.loc[:, ~df.columns.duplicated(keep="last")]

def sincronizar_desligamentos(df_movimentacoes, df_ativos, df_desligados, verbose=True):
    mov = _dedup_cols(df_movimentacoes)
    ativos = _fix_cols_ativos(df_ativos)          # <-- AQUI resolve "Cargo Especifico" duplicado
    deslig = _dedup_cols(df_desligados)

    # --- Normaliza Cadastro (Int64 compatível com NA) ---
    for df in (mov, ativos, deslig):
        df["Cadastro"] = pd.to_numeric(df["Cadastro"], errors="coerce").astype("Int64")

    # --- Garante que df_desligados tenha todas as colunas alvo ---
    for c in TARGET_COLS:
        if c not in deslig.columns:
            deslig[c] = pd.NA
    deslig = deslig[TARGET_COLS].copy()

    # --- Parse datas existentes do df_desligados ---
    deslig["Data_Adm_dt"] = _to_dt(deslig["Data_Adm"], dayfirst=True)
    deslig["Data_Dem_dt"] = _to_dt(deslig["Data_Dem"], dayfirst=True)

    # --- Filtra desligamentos nas movimentações ---
    mov["Tipo da Movimentação"] = mov["Tipo da Movimentação"].astype(str).str.strip()
    mov["Data de alteração_dt"] = _to_dt(mov["Data de alteração"], dayfirst=True)

    mov_desl = mov[mov["Tipo da Movimentação"].str.contains("deslig", case=False, na=False)].copy()

    if mov_desl.empty:
        if verbose:
            print("ℹ️ Nenhum 'Desligamento' encontrado em df_movimentacoes.")
        deslig = deslig.drop(columns=["Data_Adm_dt", "Data_Dem_dt"], errors="ignore")
        return ativos, deslig

    cols_mov_util = ["Cadastro", "Data de alteração_dt"]

    if "Cargo" in mov_desl.columns:
        cols_mov_util.append("Cargo")

    for cand in ["Abreviação UO", "Abreviação UO antiga"]:
        if cand in mov_desl.columns:
            cols_mov_util.append(cand); break
    for cand in ["Desc. UO", "Desc. UO antiga"]:
        if cand in mov_desl.columns:
            cols_mov_util.append(cand); break
    for cand in ["Desc. Cargo", "Desc. Cargo antigo"]:
        if cand in mov_desl.columns:
            cols_mov_util.append(cand); break

    mov_last = (
        mov_desl.sort_values("Data de alteração_dt")
               .groupby("Cadastro", as_index=False)
               .tail(1)[cols_mov_util]
               .rename(columns={"Data de alteração_dt": "Data_Dem_dt"})
    )

    incoming = pd.DataFrame({"Cadastro": mov_last["Cadastro"]})
    incoming["Cargo"] = mov_last["Cargo"] if "Cargo" in mov_last.columns else pd.NA

    incoming["Nome completo"] = pd.NA
    incoming["Heranca"] = pd.NA
    incoming["Local"] = pd.NA
    incoming["Data_Adm_dt"] = pd.NaT
    incoming["Data_Dem_dt"] = mov_last["Data_Dem_dt"]

    col_abrev = next((c for c in ["Abreviação UO", "Abreviação UO antiga"] if c in mov_last.columns), None)
    col_uo = next((c for c in ["Desc. UO", "Desc. UO antiga"] if c in mov_last.columns), None)
    col_cargo = next((c for c in ["Desc. Cargo", "Desc. Cargo antigo"] if c in mov_last.columns), None)

    incoming["Área"] = mov_last[col_abrev] if col_abrev else pd.NA
    incoming["Departamento"] = mov_last[col_uo] if col_uo else pd.NA
    incoming["Cargo Especifico"] = mov_last[col_cargo] if col_cargo else pd.NA

    ativos["Data_Admissão_dt"] = _to_dt(ativos.get("Data_Admissão"), dayfirst=True)

    in_ativos = incoming.merge(ativos, on="Cadastro", how="left", suffixes=("", "_a"))

    # (opcional mas recomendado) puxar também o Cargo de ativos, se veio NA em mov
    incoming["Cargo"] = _coalesce(incoming["Cargo"], in_ativos.get("Cargo_a"))

    incoming["Nome completo"] = _coalesce(incoming["Nome completo"], in_ativos.get("Nome completo_a"))
    incoming["Heranca"] = _coalesce(incoming["Heranca"], in_ativos.get("Heranca_a"))
    incoming["Cargo Especifico"] = _coalesce(incoming["Cargo Especifico"], in_ativos.get("Cargo Especifico_a"))
    incoming["Departamento"] = _coalesce(incoming["Departamento"], in_ativos.get("Departamento_a"))
    incoming["Área"] = _coalesce(incoming["Área"], in_ativos.get("Área_a"))
    incoming["Local"] = _coalesce(incoming["Local"], in_ativos.get("Local_a"))
    incoming["Data_Adm_dt"] = _coalesce(incoming["Data_Adm_dt"], in_ativos.get("Data_Admissão_dt"))

    cad_mover = incoming["Cadastro"].dropna().unique()
    ativos_out = ativos[~ativos["Cadastro"].isin(cad_mover)].reset_index(drop=True)

    base = deslig.copy()

    merged = base.merge(
        incoming,
        on="Cadastro",
        how="outer",
        suffixes=("_old", "_new")
    )

    for col in ["Cargo", "Nome completo", "Heranca", "Cargo Especifico", "Departamento", "Área", "Local"]:
        merged[col] = _coalesce(merged.get(f"{col}_old"), merged.get(f"{col}_new"))

    merged["Data_Adm_dt"] = _coalesce(merged.get("Data_Adm_dt_old"), merged.get("Data_Adm_dt_new"))

    old_dem = merged.get("Data_Dem_dt_old")
    new_dem = merged.get("Data_Dem_dt_new")
    merged["Data_Dem_dt"] = old_dem
    merged.loc[merged["Data_Dem_dt"].isna(), "Data_Dem_dt"] = new_dem
    both = old_dem.notna() & new_dem.notna()
    merged.loc[both, "Data_Dem_dt"] = pd.concat([old_dem[both], new_dem[both]], axis=1).max(axis=1)

    merged["Data_Adm"] = pd.to_datetime(merged["Data_Adm_dt"], errors="coerce").dt.strftime("%d.%m.%Y")
    merged["Data_Dem"] = pd.to_datetime(merged["Data_Dem_dt"], errors="coerce").dt.strftime("%d.%m.%Y")

    desligados_out = merged[TARGET_COLS].reset_index(drop=True)

    if verbose:
        print(f"🔎 Desligamentos (movimentações): {len(incoming)}")
        removidos = df_ativos["Cadastro"].isin(cad_mover).sum() if "Cadastro" in df_ativos.columns else 0
        print(f"🧹 Removidos de ativos (se existiam): {removidos}")
        print(f"✅ Ativos: {len(df_ativos)} -> {len(ativos_out)}")
        print(f"✅ Desligados: {len(df_desligados)} -> {len(desligados_out)}")

    ativos_out = ativos_out.drop(columns=["Data_Admissão_dt"], errors="ignore")
    return ativos_out, desligados_out


In [ ]:
# Executando a Função:
df_ativos, df_desligados = sincronizar_desligamentos(
    df_movimentacoes, df_ativos, df_desligados, verbose=True
)


In [ ]:
# Normalização mínima do 'Cadastro' (evita 123, "123", 123.0 divergirem)
def _norm_id(s):
    s = s.astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
    return s.replace({'nan': np.nan})  # volta NaN onde era NaN

for _df in (df_ronda, df_desligados, df_ativos):
    if 'Cadastro' not in _df.columns:
        raise KeyError("Faltou a coluna 'Cadastro' em um dos DataFrames.")
    _df['Cadastro'] = _norm_id(_df['Cadastro'])

# Desligado por Cadastro:
df_ronda['Desligado'] = df_ronda['Cadastro'].isin(df_desligados['Cadastro']).map({True: 'Sim', False: 'Não'})

# Preparar 'df_ativos' com nomes únicos de coluna:
deps = df_ativos[['Cadastro','Nome completo','Área','Departamento','Heranca','Cargo Especifico']].rename(
    columns={
        'Nome completo':'Nome_dep',
        'Área':'Área_dep',
        'Departamento':'Departamento_dep',
        'Heranca':'Heranca_dep',
        'Cargo Especifico':'Cargo Especifico_dep'
    }
)


In [ ]:
# Merge (left) pelo Cadastro:
df_ronda = df_ronda.merge(deps, on='Cadastro', how='left')


In [ ]:
# Dicionários (df_desligados) por Cadastro:
map_df_desligados_nome = dict(zip(df_desligados['Cadastro'], df_desligados['Nome completo']))
map_df_desligados_area = dict(zip(df_desligados['Cadastro'], df_desligados['Área']))
map_df_desligados_dept = dict(zip(df_desligados['Cadastro'], df_desligados['Departamento']))
map_df_desligados_her  = dict(zip(df_desligados['Cadastro'], df_desligados['Heranca']))
map_df_desligados_carg = dict(zip(df_desligados['Cadastro'], df_desligados['Cargo Especifico']))

# Substituir com prioridade: df_desligados > df_ativos > original:

# Nome:
df_ronda['Nome'] = (
    df_ronda['Cadastro'].map(map_df_desligados_nome)   # 1º: df_desligados…
    .fillna(df_ronda['Nome_dep'])                  # 2º: df_ativos…
    .fillna(df_ronda.get('Nome'))                  # 3º: valor original (se existir)…
)

# Área:
df_ronda['Área'] = (
    df_ronda['Cadastro'].map(map_df_desligados_area)
    .fillna(df_ronda['Área_dep'])
)

# Departamento:
df_ronda['Departamento'] = (
    df_ronda['Cadastro'].map(map_df_desligados_dept)
    .fillna(df_ronda['Departamento_dep'])
)

# Heranca:
df_ronda['Heranca'] = (
    df_ronda['Cadastro'].map(map_df_desligados_her)
    .fillna(df_ronda['Heranca_dep'])
)

# Cargo Especifico:
df_ronda['Cargo Especifico'] = (
    df_ronda['Cadastro'].map(map_df_desligados_carg)
    .fillna(df_ronda['Cargo Especifico_dep'])
)

# (Opcional) Manter as colunas *_dep para auditoria; se quiser remover, descomente abaixo:
df_ronda = df_ronda.drop(columns=['Nome_dep','Área_dep','Departamento_dep','Heranca_dep','Cargo Especifico_dep'], errors='ignore')

#### Corrigindo as colunas de Datas:

In [ ]:
# Pareseand por regex:
s = df_ronda["Data"].astype(str).str.strip()

# só datas com barra (ex: 02/09/2026)
mask_slash = s.str.match(r"^\d{1,2}/\d{1,2}/\d{4}$")

# tenta BR primeiro (dd/mm/yyyy) sem warning, porque o format é explícito
br = pd.to_datetime(s.where(mask_slash), format="%d/%m/%Y", errors="coerce")

# onde falhou, tenta US (mm/dd/yyyy) também sem warning
us = pd.to_datetime(s.where(mask_slash), format="%m/%d/%Y", errors="coerce")

# para outros formatos (ex: YYYY-MM-DD), tenta o parser padrão
other = pd.to_datetime(s.where(~mask_slash), errors="coerce")

df_ronda["Data"] = other.fillna(br.fillna(us))


In [ ]:
# Hora (minutos -> time):
df_ronda['Hora'] = pd.to_numeric(df_ronda['Hora'], errors='coerce')
df_ronda['Hora'] = pd.to_timedelta(df_ronda['Hora'], unit='m')
df_ronda['Hora'] = (pd.Timestamp('1900-01-01') + df_ronda['Hora']).dt.time

# Data (MANTER como datetime — não usar strftime aqui):
df_ronda['Data'] = pd.to_datetime(df_ronda['Data'], dayfirst=True, errors='coerce')

# Admissão/Demissão (datetime):
df_desligados['Data_Adm'] = pd.to_datetime(df_desligados['Data_Adm'], dayfirst=True, errors='coerce')
df_desligados['Data_Dem'] = pd.to_datetime(df_desligados['Data_Dem'], dayfirst=True, errors='coerce')
df_ativos['Data_Admissão'] = pd.to_datetime(df_ativos['Data_Admissão'], format='%d.%m.%Y', errors='coerce')

def fmt_dt(x):
    return x.strftime('%d/%m/%Y') if pd.notna(x) else 'NaT'

print(f"df_ronda['Data']           -> min={fmt_dt(df_ronda['Data'].min())} | max={fmt_dt(df_ronda['Data'].max())}")
print(f"df_desligados['Data_Adm']  -> min={fmt_dt(df_desligados['Data_Adm'].min())} | max={fmt_dt(df_desligados['Data_Adm'].max())}")
print(f"df_desligados['Data_Dem']  -> min={fmt_dt(df_desligados['Data_Dem'].min())} | max={fmt_dt(df_desligados['Data_Dem'].max())}")
print(f"df_ativos['Data_Admissão'] -> min={fmt_dt(df_ativos['Data_Admissão'].min())} | max={fmt_dt(df_ativos['Data_Admissão'].max())}")

## 3. Carregando os Dados:

#### Base Inteira para análises específicas:

In [ ]:
# Criando a função para a geração dos arquivos parquet:
def exportar_parquets(dfs: dict[str, pd.DataFrame], base_dir: str = r"..\project\database") -> None:
    """
    Exporta vários DataFrames para .parquet dentro de uma pasta base.
    
    Exemplo de uso:
    exportar_parquets({
        "RegistrosSemanais.parquet": df_registros_semanais,
        "RegistrosDiarios.parquet": df_diarios,
        "Almoco.parquet": df_almoco,
    })
    """
    os.makedirs(base_dir, exist_ok=True)

    for filename, df in dfs.items():
        path = os.path.join(base_dir, filename)
        df.to_parquet(path, index=False)
        print(f"✅Carregado: '{path}'")

### Desenvolvendo toda a Engenharia dos Dados em formato de Data Warehouse:

In [ ]:
# ETL "executável" (sem função principal) para construir o DW SQLite `dw_ronda.sqlite` a partir do df_ronda.
"""
✅ Foco: performance no APP (Dash) via tabelas agregadas MATERIALIZADAS (agg_*).
✅ O app abre o arquivo em READ-ONLY / immutable, sem DDL.

Pré-requisitos no Notebook:
  - df_ronda (obrigatório)
  - df_movimentacoes (opcional)
  - df_gptw (opcional)
  - df_headcount (opcional)  # se não passar, calcula automaticamente

Como usar (no fim do arquivo tem um bloco "EXECUÇÃO"):
  - ajuste SQLITE_PATH e CHUNK_SIZE
  - garanta que df_ronda exista em memória
  - rode o script/célula

Obs: Datas ficam como TEXT ISO (YYYY-MM-DD) no SQLite (ótimo p/ índice). No app, parse com pd.to_datetime.
"""

from __future__ import annotations

import sqlite3
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd

# =========================
# CONFIG (AJUSTE AQUI)
# =========================
SQLITE_PATH = Path(r"..\project\database\dw_ronda.sqlite")
CHUNK_SIZE = 250_000  # ajuste conforme RAM

# =========================
# HELPERS
# =========================
def normalize_columns_inplace(df: pd.DataFrame) -> None:
    df.columns = [c.strip() for c in df.columns]
    rename_map = {
        "Cadastro": "cadastro",
        "Nome": "nome",
        "Filial": "filial",
        "Data": "data",
        "Hora": "hora",
        "Dir": "dir",
        "Tipo": "tipo",
        "Acesso": "acesso",
        "Planta": "planta",
        "Coletor": "coletor",
        "Descricao Coletor": "descricao_coletor",
        "Descrição Coletor": "descricao_coletor",
        "Uso Acesso": "uso_acesso",
        "Cargo": "cargo",
        "Desligado": "desligado",
        "Área": "area",
        "Area": "area",
        "Departamento": "departamento",
        "Heranca": "heranca",
        "Herança": "heranca",
        "Cargo Especifico": "cargo_especifico",
        "Cargo Específico": "cargo_especifico",
    }
    df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns}, inplace=True)

def _clean_date_time(chunk: pd.DataFrame) -> tuple[pd.Series, pd.Series, pd.Series]:
    # data_iso
    data_dt = pd.to_datetime(chunk["data"], errors="coerce")
    data_iso = pd.Series(data_dt.dt.strftime("%Y-%m-%d"), dtype="string")

    # hora_iso (extrai HH:MM(:SS)? e normaliza)
    hora_str = chunk["hora"].astype("string").str.strip()
    hora_extr = hora_str.str.extract(r"(\d{1,2}:\d{2}(?::\d{2})?)", expand=False)
    hora_clean = hora_extr.where(
        hora_extr.str.contains(r":\d{2}:\d{2}$", regex=True),
        hora_extr + ":00",
    )
    hora_clean = hora_clean.str.replace(r"^(\d):", r"0\1:", regex=True)
    hora_dt = pd.to_datetime(hora_clean, format="%H:%M:%S", errors="coerce")
    hora_iso = pd.Series(hora_dt.dt.strftime("%H:%M:%S"), dtype="string")

    # ts_iso
    ts = pd.to_datetime((data_iso + " " + hora_iso), format="%Y-%m-%d %H:%M:%S", errors="coerce")
    ts_iso = pd.Series(ts.dt.strftime("%Y-%m-%d %H:%M:%S"), dtype="string")

    return data_iso, hora_iso, ts_iso

# =========================
# DDL (MATERIALIZADO)
# =========================
DDL_SQL = r"""
PRAGMA foreign_keys = ON;

-- Base (flat)
DROP TABLE IF EXISTS evento_acesso_flat;
CREATE TABLE evento_acesso_flat (
  cadastro            TEXT,
  data                TEXT,  -- YYYY-MM-DD
  hora                TEXT,  -- HH:MM:SS
  ts                  TEXT,  -- YYYY-MM-DD HH:MM:SS
  filial              TEXT,
  planta              TEXT,
  coletor             TEXT,  -- UPPER no ETL
  descricao_coletor   TEXT,
  dir                 TEXT,  -- UPPER no ETL
  tipo                TEXT,
  acesso              TEXT,
  uso_acesso          TEXT
);

-- Dim colaborador (último estado por cadastro)
DROP TABLE IF EXISTS dim_colaborador;
CREATE TABLE dim_colaborador (
  cadastro            TEXT PRIMARY KEY,
  nome                TEXT,
  cargo               TEXT,
  cargo_especifico    TEXT,
  area                TEXT,
  departamento        TEXT,
  heranca             TEXT,
  desligado           TEXT,
  created_at          TEXT DEFAULT (datetime('now'))
) WITHOUT ROWID;

-- Agregados p/ app
DROP TABLE IF EXISTS agg_diarios_unico;
CREATE TABLE agg_diarios_unico (
  cadastro TEXT,
  data     TEXT,
  planta   TEXT,
  PRIMARY KEY (cadastro, data)
) WITHOUT ROWID;

DROP TABLE IF EXISTS agg_registros_semanais;
CREATE TABLE agg_registros_semanais (
  cadastro TEXT,
  ano      INTEGER,
  semana_por_ano INTEGER,
  data     TEXT,
  contagem_presenca INTEGER,
  PRIMARY KEY (cadastro, ano, semana_por_ano)
) WITHOUT ROWID;

DROP TABLE IF EXISTS agg_semana_enriquecida;
CREATE TABLE agg_semana_enriquecida (
  cadastro TEXT,
  nome     TEXT,
  cargo    TEXT,
  cargo_especifico TEXT,
  area     TEXT,
  departamento TEXT,
  heranca  TEXT,
  desligado TEXT,
  ano      INTEGER,
  semana_por_ano INTEGER,
  data     TEXT,
  contagem_presenca INTEGER
);

DROP TABLE IF EXISTS agg_almoco;
CREATE TABLE agg_almoco (
  cadastro TEXT,
  data     TEXT,
  hora     TEXT,
  dir      TEXT,
  planta   TEXT,
  coletor  TEXT,
  descricao_coletor TEXT
);

-- Opcionais
DROP TABLE IF EXISTS gptw;
CREATE TABLE gptw (area TEXT, y2025 REAL, y2024 REAL, y2023 REAL);

DROP TABLE IF EXISTS movimentacoes;
CREATE TABLE movimentacoes (
  cadastro TEXT,
  abreviacao_uo_antiga TEXT,
  desc_uo_antiga TEXT,
  desc_cargo_antigo TEXT,
  abreviacao_uo TEXT,
  desc_uo TEXT,
  desc_cargo TEXT,
  tipo_movimentacao TEXT,
  data_alteracao TEXT
);

DROP TABLE IF EXISTS headcount;
CREATE TABLE headcount (area TEXT, y2023 INTEGER, y2024 INTEGER, y2025 INTEGER);

-- Views PT-BR (contratos do app)
DROP VIEW IF EXISTS vw_colaboradores;
CREATE VIEW vw_colaboradores AS
SELECT
  cadastro            AS "Cadastro",
  nome                AS "Nome",
  cargo               AS "Cargo",
  cargo_especifico    AS "Cargo Especifico",
  area                AS "Área",
  departamento        AS "Departamento",
  heranca             AS "Heranca",
  desligado           AS "Desligado"
FROM dim_colaborador;

DROP VIEW IF EXISTS vw_df_diarios_unico;
CREATE VIEW vw_df_diarios_unico AS
SELECT cadastro AS "Cadastro", planta AS "Planta", data AS "Data"
FROM agg_diarios_unico;

DROP VIEW IF EXISTS vw_df_diarios;
CREATE VIEW vw_df_diarios AS
SELECT cadastro AS "Cadastro", planta AS "Planta", data AS "Data"
FROM agg_diarios_unico;

DROP VIEW IF EXISTS vw_df_registros_semanais;
CREATE VIEW vw_df_registros_semanais AS
SELECT
  cadastro AS "Cadastro",
  ano AS "Ano",
  semana_por_ano AS "Semana por Ano",
  data AS "Data",
  contagem_presenca AS "Contagem Presença"
FROM agg_registros_semanais;

DROP VIEW IF EXISTS vw_semana_enriquecida;
CREATE VIEW vw_semana_enriquecida AS
SELECT
  cadastro AS "Cadastro",
  nome AS "Nome",
  cargo AS "Cargo",
  cargo_especifico AS "Cargo Especifico",
  area AS "Área",
  departamento AS "Departamento",
  heranca AS "Heranca",
  desligado AS "Desligado",
  ano AS "Ano",
  semana_por_ano AS "Semana por Ano",
  data AS "Data",
  contagem_presenca AS "Contagem Presença"
FROM agg_semana_enriquecida;

DROP VIEW IF EXISTS vw_df_almoco;
CREATE VIEW vw_df_almoco AS
SELECT
  cadastro AS "Cadastro",
  data AS "Data",
  hora AS "Hora",
  dir AS "Dir",
  planta AS "Planta",
  coletor AS "Coletor",
  descricao_coletor AS "Descricao Coletor"
FROM agg_almoco;

DROP VIEW IF EXISTS vw_gptw;
CREATE VIEW vw_gptw AS
SELECT area AS "Área", CAST(y2025 AS REAL) AS "2025", CAST(y2024 AS REAL) AS "2024", CAST(y2023 AS REAL) AS "2023"
FROM gptw;

DROP VIEW IF EXISTS vw_movimentacoes;
CREATE VIEW vw_movimentacoes AS
SELECT
  cadastro AS "Cadastro",
  abreviacao_uo_antiga AS "Abreviação UO antiga",
  desc_uo_antiga AS "Desc. UO antiga",
  desc_cargo_antigo AS "Desc. Cargo antigo",
  abreviacao_uo AS "Abreviação UO",
  desc_uo AS "Desc. UO",
  desc_cargo AS "Desc. Cargo",
  tipo_movimentacao AS "Tipo da Movimentação",
  data_alteracao AS "Data de alteração"
FROM movimentacoes;

DROP VIEW IF EXISTS vw_headcount;
CREATE VIEW vw_headcount AS
SELECT area AS "Área", CAST(y2023 AS INTEGER) AS "2023", CAST(y2024 AS INTEGER) AS "2024", CAST(y2025 AS INTEGER) AS "2025"
FROM headcount;
"""

# =========================
# EXECUÇÃO (SEM FUNÇÃO)
# =========================

# Espera-se que df_ronda exista no ambiente.
# df_movimentacoes/df_gptw/df_headcount são opcionais; se não existirem, deixa como None.
df_movimentacoes = globals().get("df_movimentacoes", None)
df_gptw = globals().get("df_gptw", None)
df_headcount = globals().get("df_headcount", None)

assert "df_ronda" in globals(), "df_ronda não encontrado no ambiente. Carregue df_ronda antes de rodar o ETL."
df_ronda = globals()["df_ronda"]

SQLITE_PATH.parent.mkdir(parents=True, exist_ok=True)
if SQLITE_PATH.exists():
    SQLITE_PATH.unlink()

normalize_columns_inplace(df_ronda)

required = [
    "cadastro", "nome", "filial", "data", "hora", "dir", "tipo", "acesso", "planta",
    "coletor", "descricao_coletor", "uso_acesso", "cargo", "desligado", "area",
    "departamento", "heranca", "cargo_especifico",
]

# Para DIM: manter “último estado” por cadastro sem guardar 35M linhas em staging
# Guardamos em dict: cadastro -> (ts, nome, cargo, cargo_especifico, area, departamento, heranca, desligado)
dim_latest: dict[str, tuple[str, str, str, str, str, str, str, str]] = {}

conn = sqlite3.connect(SQLITE_PATH)
try:
    # Bulk load PRAGMAs
    conn.execute("PRAGMA page_size = 32768;")
    conn.execute("PRAGMA journal_mode = OFF;")
    conn.execute("PRAGMA synchronous = OFF;")
    conn.execute("PRAGMA temp_store = MEMORY;")
    conn.execute("PRAGMA cache_size = -600000;")  # ~600MB
    conn.execute("PRAGMA locking_mode = EXCLUSIVE;")
    conn.execute("PRAGMA foreign_keys = ON;")

    conn.executescript(DDL_SQL)
    cur = conn.cursor()

    insert_flat = """INSERT INTO evento_acesso_flat (
        cadastro,data,hora,ts,filial,planta,coletor,descricao_coletor,dir,tipo,acesso,uso_acesso
    ) VALUES (?,?,?,?,?,?,?,?,?,?,?,?);"""

    n = len(df_ronda)
    for start in range(0, n, CHUNK_SIZE):
        end = min(start + CHUNK_SIZE, n)
        chunk = df_ronda.iloc[start:end].reindex(columns=required, fill_value=pd.NA).copy()
        chunk.replace({"nan": pd.NA, "NaN": pd.NA, "None": pd.NA, "": pd.NA}, inplace=True)

        data_iso, hora_iso, ts_iso = _clean_date_time(chunk)

        # Flat (mínimo necessário p/ agregações + filtros)
        flat_df = pd.DataFrame({
            "cadastro": chunk["cadastro"].astype("string").str.strip(),
            "data": data_iso,
            "hora": hora_iso,
            "ts": ts_iso,
            "filial": chunk["filial"].astype("string").str.strip(),
            "planta": chunk["planta"].astype("string").str.strip(),
            "coletor": chunk["coletor"].astype("string").str.strip().str.upper(),
            "descricao_coletor": chunk["descricao_coletor"].astype("string").str.strip(),
            "dir": chunk["dir"].astype("string").str.strip().str.upper(),
            "tipo": chunk["tipo"].astype("string").str.strip(),
            "acesso": chunk["acesso"].astype("string").str.strip(),
            "uso_acesso": chunk["uso_acesso"].astype("string").str.strip(),
        })

        # ✅ bind-safe: pandas.NA -> None
        flat_obj = flat_df.astype("object").where(flat_df.notna(), None)

        conn.execute("BEGIN;")
        cur.executemany(insert_flat, flat_obj.itertuples(index=False, name=None))
        conn.commit()

        # Atualiza DIM (último estado por cadastro) usando ts
        # Faz isso em pandas para evitar staging gigante + window function em 35M.
        dim_chunk = pd.DataFrame({
            "cadastro": chunk["cadastro"].astype("string").str.strip(),
            "ts": ts_iso,
            "nome": chunk["nome"].astype("string").str.strip(),
            "cargo": chunk["cargo"].astype("string").str.strip(),
            "cargo_especifico": chunk["cargo_especifico"].astype("string").str.strip(),
            "area": chunk["area"].astype("string").str.strip(),
            "departamento": chunk["departamento"].astype("string").str.strip(),
            "heranca": chunk["heranca"].astype("string").str.strip(),
            "desligado": chunk["desligado"].astype("string").str.strip(),
        }).astype("object").where(lambda d: d.notna(), None)

        # remove cadastros/ts vazios
        dim_chunk = dim_chunk[(dim_chunk["cadastro"].notna()) & (dim_chunk["ts"].notna())]

        # pega o registro de maior ts por cadastro no chunk (ts ISO ordena como string)
        dim_chunk.sort_values(["cadastro", "ts"], inplace=True)
        last_per_cad = dim_chunk.groupby("cadastro", as_index=False).tail(1)

        for row in last_per_cad.itertuples(index=False):
            cad = row.cadastro
            ts = row.ts
            prev = dim_latest.get(cad)
            if prev is None or ts > prev[0]:
                dim_latest[cad] = (ts, row.nome, row.cargo, row.cargo_especifico, row.area, row.departamento, row.heranca, row.desligado)

    # Índices no flat (acelera agregações e filtros)
    conn.executescript("""
        CREATE INDEX IF NOT EXISTS ix_flat_cad_data_ts   ON evento_acesso_flat (cadastro, data, ts);
        CREATE INDEX IF NOT EXISTS ix_flat_data          ON evento_acesso_flat (data);
        CREATE INDEX IF NOT EXISTS ix_flat_planta        ON evento_acesso_flat (planta);
        CREATE INDEX IF NOT EXISTS ix_flat_tipo          ON evento_acesso_flat (tipo);
        CREATE INDEX IF NOT EXISTS ix_flat_dir           ON evento_acesso_flat (dir);
        CREATE INDEX IF NOT EXISTS ix_flat_coletor       ON evento_acesso_flat (coletor);
    """)

    # Carrega DIM no SQLite
    conn.execute("BEGIN;")
    conn.execute("DELETE FROM dim_colaborador;")
    conn.executemany(
        "INSERT INTO dim_colaborador(cadastro,nome,cargo,cargo_especifico,area,departamento,heranca,desligado) VALUES (?,?,?,?,?,?,?,?);",
        [(cad, nome, cargo, cargo_esp, area, depto, her, desl) for cad, (ts, nome, cargo, cargo_esp, area, depto, her, desl) in dim_latest.items()]
    )
    conn.commit()
    conn.executescript("""
        CREATE INDEX IF NOT EXISTS ix_dim_area   ON dim_colaborador (area);
        CREATE INDEX IF NOT EXISTS ix_dim_depto  ON dim_colaborador (departamento);
    """)

    # =========================
    # AGREGAÇÕES MATERIALIZADAS
    # =========================

    # Presença válida + diários único (último ts do dia por cadastro)
    conn.executescript("""
        DROP TABLE IF EXISTS tmp_presenca_validos;
        CREATE TABLE tmp_presenca_validos AS
        SELECT
          cadastro,
          data AS data_dia,
          ts,
          planta
        FROM evento_acesso_flat
        WHERE
          cadastro IS NOT NULL
          AND data IS NOT NULL
          AND ts IS NOT NULL
          AND planta IS NOT NULL
          AND planta <> '999'
          AND tipo IS NOT NULL
          AND tipo <> '0';

        CREATE INDEX IF NOT EXISTS ix_tmp_pres_cad_data_ts ON tmp_presenca_validos (cadastro, data_dia, ts);

        -- ts máximo por cadastro+dia
        DROP TABLE IF EXISTS tmp_last;
        DROP TABLE IF EXISTS tmp_last_rid;
        CREATE TABLE tmp_last AS
        SELECT cadastro, data_dia, MAX(ts) AS ts_max
        FROM tmp_presenca_validos
        GROUP BY cadastro, data_dia;

        CREATE INDEX IF NOT EXISTS ix_tmp_last ON tmp_last (cadastro, data_dia);

        -- Empate no mesmo ts: pega a MAIOR rowid (determinístico) e evita duplicar PK
        DROP TABLE IF EXISTS tmp_last_rid;
        CREATE TABLE tmp_last_rid AS
        SELECT
          p.cadastro,
          p.data_dia,
          MAX(p.rowid) AS rid
        FROM tmp_presenca_validos p
        JOIN tmp_last l
          ON p.cadastro = l.cadastro
         AND p.data_dia = l.data_dia
         AND p.ts = l.ts_max
        GROUP BY p.cadastro, p.data_dia;

        CREATE INDEX IF NOT EXISTS ix_tmp_last_rid ON tmp_last_rid (cadastro, data_dia);

        INSERT INTO agg_diarios_unico (cadastro, data, planta)
        SELECT
          p.cadastro,
          p.data_dia AS data,
          p.planta
        FROM tmp_presenca_validos p
        JOIN tmp_last_rid r
          ON p.cadastro = r.cadastro
         AND p.data_dia = r.data_dia
         AND p.rowid = r.rid;

        CREATE INDEX IF NOT EXISTS ix_diarios_data   ON agg_diarios_unico (data);
        CREATE INDEX IF NOT EXISTS ix_diarios_planta ON agg_diarios_unico (planta);
    """)

    # Semana ISO (ancora quinta)
    conn.executescript("""
        WITH base AS (
          SELECT
            cadastro,
            planta,
            data AS data_dia,
            date(data, '-3 days', 'weekday 4') AS iso_anchor
          FROM agg_diarios_unico
        )
        INSERT INTO agg_registros_semanais (cadastro, ano, semana_por_ano, data, contagem_presenca)
        SELECT
          cadastro,
          CAST(strftime('%Y', iso_anchor) AS INTEGER) AS ano,
          CAST(strftime('%W', iso_anchor) AS INTEGER) + 1 AS semana_por_ano,
          date(iso_anchor, '-3 days') AS data,
          COUNT(*) AS contagem_presenca
        FROM base
        GROUP BY cadastro, ano, semana_por_ano, date(iso_anchor, '-3 days');

        CREATE INDEX IF NOT EXISTS ix_semana_data      ON agg_registros_semanais (data);
        CREATE INDEX IF NOT EXISTS ix_semana_ano_sem   ON agg_registros_semanais (ano, semana_por_ano);
    """)

    # Semana enriquecida
    conn.executescript("""
        INSERT INTO agg_semana_enriquecida (
          cadastro, nome, cargo, cargo_especifico, area, departamento, heranca, desligado,
          ano, semana_por_ano, data, contagem_presenca
        )
        SELECT
          s.cadastro,
          c.nome,
          c.cargo,
          c.cargo_especifico,
          c.area,
          c.departamento,
          c.heranca,
          c.desligado,
          s.ano,
          s.semana_por_ano,
          s.data,
          s.contagem_presenca
        FROM agg_registros_semanais s
        LEFT JOIN dim_colaborador c
          ON c.cadastro = s.cadastro;

        CREATE INDEX IF NOT EXISTS ix_semana_enr_area  ON agg_semana_enriquecida (area);
        CREATE INDEX IF NOT EXISTS ix_semana_enr_data  ON agg_semana_enriquecida (data);
    """)

    # Almoço (dir/coletor já UPPER)
    conn.executescript("""
        INSERT INTO agg_almoco (cadastro, data, hora, dir, planta, coletor, descricao_coletor)
        SELECT
          cadastro,
          data AS data,
          hora,
          dir,
          planta,
          coletor,
          descricao_coletor
        FROM evento_acesso_flat
        WHERE
          data IS NOT NULL
          AND hora IS NOT NULL
          AND (
            (
              dir = 'E'
              AND coletor LIKE '%REF%'
              AND coletor NOT LIKE '%GRILL%'
              AND coletor NOT LIKE '%GRIL%'
            )
            OR
            (
              dir = 'S'
              AND coletor LIKE '%PORT%'
              AND hora >= '11:00:00' AND hora <= '14:00:00'
            )
          );

        CREATE INDEX IF NOT EXISTS ix_almoco_cad_data ON agg_almoco (cadastro, data);
        CREATE INDEX IF NOT EXISTS ix_almoco_data     ON agg_almoco (data);
    """)

    # =========================
    # CARGAS OPCIONAIS
    # =========================
    if isinstance(df_gptw, pd.DataFrame) and not df_gptw.empty:
        g = df_gptw.copy()
        g.columns = [c.strip() for c in g.columns]
        if "Área" in g.columns:
            g.rename(columns={"Área": "area"}, inplace=True)
        for y in ("2025", "2024", "2023"):
            if y in g.columns:
                g[y] = pd.to_numeric(g[y].replace({"**": np.nan}), errors="coerce")
        g2 = g[["area", "2025", "2024", "2023"]].rename(columns={"2025": "y2025", "2024": "y2024", "2023": "y2023"})
        g2 = g2.astype("object").where(g2.notna(), None)
        conn.execute("DELETE FROM gptw;")
        conn.executemany("INSERT INTO gptw(area,y2025,y2024,y2023) VALUES (?,?,?,?);", g2.itertuples(index=False, name=None))

    if isinstance(df_movimentacoes, pd.DataFrame) and not df_movimentacoes.empty:
        m = df_movimentacoes.copy()
        m.columns = [c.strip() for c in m.columns]
        rename = {
            "Cadastro": "cadastro",
            "Abreviação UO antiga": "abreviacao_uo_antiga",
            "Desc. UO antiga": "desc_uo_antiga",
            "Desc. Cargo antigo": "desc_cargo_antigo",
            "Abreviação UO": "abreviacao_uo",
            "Desc. UO": "desc_uo",
            "Desc. Cargo": "desc_cargo",
            "Tipo da Movimentação": "tipo_movimentacao",
            "Data de alteração": "data_alteracao",
        }
        m.rename(columns={k: v for k, v in rename.items() if k in m.columns}, inplace=True)
        if "data_alteracao" in m.columns:
            m["data_alteracao"] = pd.to_datetime(m["data_alteracao"], errors="coerce").dt.strftime("%Y-%m-%d")

        cols = ["cadastro","abreviacao_uo_antiga","desc_uo_antiga","desc_cargo_antigo","abreviacao_uo","desc_uo","desc_cargo","tipo_movimentacao","data_alteracao"]
        for c in cols:
            if c not in m.columns:
                m[c] = pd.NA
        m2 = m[cols].astype("object").where(m[cols].notna(), None)

        conn.execute("DELETE FROM movimentacoes;")
        conn.executemany(
            """INSERT INTO movimentacoes(
                cadastro,abreviacao_uo_antiga,desc_uo_antiga,desc_cargo_antigo,
                abreviacao_uo,desc_uo,desc_cargo,tipo_movimentacao,data_alteracao
            ) VALUES (?,?,?,?,?,?,?,?,?);""",
            m2.itertuples(index=False, name=None),
        )
        conn.executescript("""
            CREATE INDEX IF NOT EXISTS ix_mov_cadastro ON movimentacoes(cadastro);
            CREATE INDEX IF NOT EXISTS ix_mov_data ON movimentacoes(data_alteracao);
        """)

    # =========================
    # HEADCOUNT (FEITO NO ETL)
    # =========================
    if isinstance(df_headcount, pd.DataFrame) and not df_headcount.empty:
        h = df_headcount.copy()
        h.columns = [c.strip() for c in h.columns]
        if "Área" in h.columns:
            h.rename(columns={"Área": "area"}, inplace=True)
        for y in ("2023", "2024", "2025"):
            if y in h.columns:
                h[y] = pd.to_numeric(h[y], errors="coerce")
        h2 = h[["area", "2023", "2024", "2025"]].rename(columns={"2023": "y2023", "2024": "y2024", "2025": "y2025"})
        h2 = h2.astype("object").where(h2.notna(), None)
        conn.execute("DELETE FROM headcount;")
        conn.executemany("INSERT INTO headcount(area,y2023,y2024,y2025) VALUES (?,?,?,?);", h2.itertuples(index=False, name=None))
    else:
        conn.executescript("""
            DELETE FROM headcount;

            DROP TABLE IF EXISTS tmp_cad_ano;
            CREATE TABLE tmp_cad_ano AS
            SELECT DISTINCT cadastro, ano
            FROM agg_registros_semanais;

            CREATE INDEX IF NOT EXISTS ix_tmp_cad_ano ON tmp_cad_ano (ano, cadastro);

            INSERT INTO headcount(area, y2023, y2024, y2025)
            WITH base AS (
              SELECT
                COALESCE(NULLIF(TRIM(c.area), ''), '(Sem Área)') AS area,
                t.ano AS ano,
                t.cadastro AS cadastro
              FROM tmp_cad_ano t
              JOIN dim_colaborador c
                ON c.cadastro = t.cadastro
            )
            SELECT
              area,
              COUNT(DISTINCT CASE WHEN ano = 2023 THEN cadastro END) AS y2023,
              COUNT(DISTINCT CASE WHEN ano = 2024 THEN cadastro END) AS y2024,
              COUNT(DISTINCT CASE WHEN ano = 2025 THEN cadastro END) AS y2025
            FROM base
            GROUP BY area
            ORDER BY area;

            DROP TABLE IF EXISTS tmp_cad_ano;
        """)

    # Limpeza + estatísticas
    conn.executescript("""
        DROP TABLE IF EXISTS tmp_presenca_validos;
        DROP TABLE IF EXISTS tmp_last;
        DROP TABLE IF EXISTS tmp_last_rid;
    """)
    conn.execute("ANALYZE;")
    try:
        conn.execute("PRAGMA optimize;")
    except Exception:
        pass

    # Finaliza banco para leitura (sem WAL)
    try:
        conn.execute("PRAGMA journal_mode = DELETE;")
        conn.execute("PRAGMA synchronous = NORMAL;")
    except Exception:
        pass
    conn.execute("PRAGMA locking_mode = NORMAL;")

finally:
    conn.close()

print("✅ DW gerado em:", SQLITE_PATH)
